# Agent change validated by tests — execution evidence

Runs every assertion in `tests/test_operational_schema.sql` against the live branch and shows the pass/fail output, validating the agent's migrations (001 + 002). All tests return TRUE.

In [1]:
import sys, textwrap
sys.path.insert(0, ".")
from lib import lb
conn = lb.connect("geniebandits-dev")
conn.autocommit = True
cur = conn.cursor()

def show(sql, title=None):
    if title: print(f"### {title}")
    print(textwrap.dedent(sql).strip())
    cur.execute(sql)
    if cur.description:
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
        print("-> " + " | ".join(cols))
        for r in rows:
            print("   " + " | ".join(str(x) for x in r))
        print(f"({len(rows)} row(s))\n")
    else:
        print(f"-> OK ({cur.rowcount} affected)\n")

In [2]:
import re
sql_text = open("tests/test_operational_schema.sql").read()
tests = []
for block in re.split(r"--\s*name:\s*", sql_text)[1:]:
    name, _, rest = block.partition("\n")
    body = "\n".join(l for l in rest.splitlines() if not l.strip().startswith("--")).strip()
    if body: tests.append((name.strip(), body))
passed = 0
for name, body in tests:
    cur.execute(body); ok = bool(cur.fetchone()[0]); passed += ok
    print(("PASS" if ok else "FAIL"), name)
print(f"\n{passed}/{len(tests)} tests passed")

PASS fk_no_orphan_stores
PASS fk_no_orphan_products
PASS audit_trigger_populated


PASS hybrid_indexes_present
PASS embedding_coverage_complete
PASS check_constraint_rejects_bad_move

6/6 tests passed
